<a href="https://colab.research.google.com/github/rahilkhan-acadmic/APAIML-GradedMiniProject/blob/develop/capstone/Phase5_API_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import json
import pandas as pd
import datetime
import hashlib

app = FastAPI(title="Claim Outcome Prediction API", version="1.0.0")

# Load artifacts
model = joblib.load('random_forest_model.joblib')
with open('feature_schema.json', 'r') as f:
    feature_schema = json.load(f)

# Request Schema Validation
class ClaimRequest(BaseModel):
    age: int
    chronic_flag: int
    billed_amount: float
    approved_amount: float
    payment_days: float
    amount_difference: float
    approval_ratio: float
    days_between_visit_and_billing: int
    length_of_stay_days: float
    gender_M: bool
    city_Chennai: bool
    city_Delhi: bool
    city_Hyderabad: bool
    city_Mumbai: bool
    city_Pune: bool
    insurance_provider_HealthPlus: bool
    insurance_provider_MediCareX: bool
    insurance_provider_SecureLife: bool
    department_ER: bool
    department_General: bool
    department_ICU: bool
    department_Neurology: bool
    department_Orthopedics: bool
    visit_type_ICU: bool
    visit_type_OPD: bool
    avg_days_between_visits: float

@app.get("/health")
def health_check():
    return {"status": "healthy", "timestamp": datetime.datetime.now().isoformat()}

@app.post("/predict")
def predict(request: ClaimRequest):
    try:
        # Convert request to DataFrame
        input_data = pd.DataFrame([request.model_dump()])

        # Log input feature hash for auditability
        feature_hash = hashlib.sha256(str(request.model_dump()).encode()).hexdigest()

        # Ensure column order matches feature schema
        input_data = input_data[feature_schema]

        # Generate prediction
        prediction_idx = int(model.predict(input_data)[0])
        mapping = {0: "Paid", 1: "Pending", 2: "Rejected"}
        prediction_label = mapping.get(prediction_idx, "Unknown")

        # Log Prediction
        log_entry = {
            "timestamp": datetime.datetime.now().isoformat(),
            "model_version": "1.0.0",
            "input_hash": feature_hash,
            "prediction": prediction_label
        }
        print(f"LOG: {log_entry}")

        return log_entry
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
